# Daste, Pierré et al. (2025), passive odor experiments, from dandiset 000785

Paper: Daste, Pierré et al., "Projection-specific Routing of Odor Information in the Olfactory
Cortex", bioRxiv preprint doi:10.64898/2025.12.12.694045 (version 1). The claims file
(`claims/000785/10.64898-2025.12.12.694045.yaml`) was written from the preprint's full text
flattened from its JATS XML: main text, Methods and supplementary legends, without the contents
of Tables 2 to 8.

Dataset: DANDI:000785, version **0.260520.1603** (doi:10.48324/dandi.000785/0.260520.1603). The
paper cites no version. Two versions are published, 0.260519.1753 and 0.260520.1603; they hold
the same 61 files with the same blobs (checked by asset path and blob id through the DANDI API
before the analysis), so the most recent one is pinned. The files are streamed, never
downloaded.

This notebook covers the counts of neurons and the passive experiments (identity,
concentration and 30 Hz datasets). The Go/NoGo experiment and the tracked neurons are in
`reproduce_daste_2025_gng.ipynb`.

| Claim | Tested here |
|---|---|
| dataset-neuron-counts | tested |
| fast-dataset-counts | tested |
| figs2b-labeling-similar | tested |
| mlp-projection-f1, transformer-projection-f1, fig1g-attention-timing, fig1f-latent-separation | blocked: the hyperparameters (Tables 5, 6, 8) are missing from the paper's text |
| fig2b-lifetime-sparseness-similar, fig2c-mutual-information, fig2d-population-sparseness, fig2-proportion-responding, fig2e-reliability | tested (identity dataset) |
| fig2f-latency, fig2g-early-peaks | tested (30 Hz dataset) |
| fig2h-identity-decoding-5s, fig2i-identity-decoding-early, fig2j-identity-decoding-peaks-earlier, fig2k-mpfc-late-identity | tested (identity dataset) |
| fig3bc-concentration-tuning, fig3de-concentration-decoding, figs5g-pairwise-concentration, figs5-sparseness-concentration | tested (concentration dataset) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from scipy import stats, signal
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import pcx

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
S = pcx.survey_all()
print("dandiset", pcx.DANDISET, "version", pcx.VERSION, "files", len(S))

## Data check

Every file's metadata, counts and timing, read without the traces. The imaging rate stated in
the metadata is compared with the rate from the plane-0 timestamps.

In [ ]:
S["subject"] = S.subject.astype(str)
cols = ["experiment_name", "projection_area", "subject", "date", "n_planes", "n_planes_without_red",
        "n_cells", "n_cells_kept_planes", "n_red", "n_red_not_cell", "n_trials", "n_bad",
        "imaging_rate_meta", "fs_timestamps", "n_frames", "has_muse"]
print(S.sort_values(["experiment_name", "subject", "date"])[cols].to_string(index=False))
print()
print(S.groupby("experiment_name").agg(files=("path", "size"), mice=("subject", "nunique"),
                                       OB_mice=("projection_area", lambda x: x.eq("OB").sum())).to_string())
print("subjects in the dandiset:", S.subject.nunique(),
      "| OB-tagged mice:", S[S.projection_area == "OB"].subject.nunique(),
      "| mPFC-tagged mice:", S[S.projection_area == "mPFC"].subject.nunique())
bad_rate = S[(S.imaging_rate_meta - S.fs_timestamps).abs() > 0.5]
print(f"files whose stated imaging rate differs from the timestamps by > 0.5 Hz: {len(bad_rate)}")
print(bad_rate[["path", "imaging_rate_meta", "fs_timestamps"]].to_string(index=False))
print("red cells that are not suite2p cells:", int(S.n_red_not_cell.sum()))
print("red-cell table rows equal to red suite2p cells in every file:", bool((S.n_redcell_table == S.n_red).all()))

## dataset-neuron-counts

> The passive identity dataset holds 81 OB-p, 84 mPFC-p and 1978 untagged neurons from 10
> mice; the concentration dataset 87 OB-p, 102 mPFC-p and 2232 untagged neurons from 10 mice;
> the Go/NoGo dataset 7 mice (4 OB-p, 3 mPFC-p tagged) with, per session, 95 +/- 2 OB-p,
> 87 +/- 8 mPFC-p and 1690 +/- 35 untagged neurons.

Counts of suite2p cells by tag, as the code's `load_experiment` builds them: planes without a red
cell are skipped (`compute_dFF0`). The count over all planes is shown beside it. For Go/NoGo
the day of a session is its rank among the experiment's dates (the code's `day_num`), day 0 the
passive day; "per session" is read as (a) the total over the mice of one task day, averaged over
days 1 to 3, or (b) over days 0 to 3. The paper has 3 mPFC-p Go/NoGo mice and the dandiset 4, so
the counts are given with all mice and without sub-668, the mPFC mouse without tracking
identifiers.

In [ ]:
def counts(df, cellcol):
    g = df.groupby("projection_area").agg(red=("n_red", "sum"), cells=(cellcol, "sum"), mice=("subject", "nunique"))
    return dict(OB=int(g.loc["OB", "red"]), mPFC=int(g.loc["mPFC", "red"]),
                untagged=int(g.cells.sum() - g.red.sum()), mice=int(g.mice.sum()))

rows = []
for exp in ["identity_dataset", "concentration_dataset"]:
    d = S[S.experiment_name == exp]
    for cc, lab in [("n_cells_kept_planes", "code rule (planes with red cells)"), ("n_cells", "all planes")]:
        rows.append(dict(experiment=exp, rule=lab, **counts(d, cc)))
print(pd.DataFrame(rows).to_string(index=False))
print("paper: identity 81 / 84 / 1978 (10 mice); concentration 87 / 102 / 2232 (10 mice)")

G = S[S.experiment_name.str.startswith("GNG")].copy()
G["day"] = G.groupby("experiment_name").date.rank(method="dense").astype(int) - 1
G["untagged"] = G.n_cells_kept_planes - G.n_red
print("\nGo/NoGo sessions per mouse and day")
print(G.pivot_table(index=["projection_area", "subject"], columns="day", values="n_red", aggfunc="sum").to_string())
gng_rows = []
for excl in [False, True]:
    d = G[G.subject != "668"] if excl else G
    per_day = d.groupby(["day", "projection_area"]).n_red.sum().unstack()
    per_day["untagged"] = d.groupby("day").untagged.sum()
    for reading, days in [("a: days 1-3", [1, 2, 3]), ("b: days 0-3", [0, 1, 2, 3])]:
        x = per_day.loc[days]
        gng_rows.append(dict(mice="without sub-668" if excl else "all 8", reading=reading,
                             OB_mean=x.OB.mean(), OB_sd=x.OB.std(), mPFC_mean=x.mPFC.mean(), mPFC_sd=x.mPFC.std(),
                             untagged_mean=x.untagged.mean(), untagged_sd=x.untagged.std(),
                             n_mice=d.subject.nunique()))
gng_counts = pd.DataFrame(gng_rows)
print(gng_counts.round(1).to_string(index=False))
print("paper: 95 +/- 2 OB-p, 87 +/- 8 mPFC-p, 1690 +/- 35 untagged per session, 7 mice")
exp_vals = [81, 84, 1978, 87, 102, 2232, 95, 87, 1690]
tols = [8.1, 8.4, 197.8, 8.7, 10.2, 223.2, 2, 8, 35]
code = pd.DataFrame(rows)
code = code[code.rule.str.startswith("code")]
for _, r in gng_counts.iterrows():
    vals = list(code.iloc[0][["OB", "mPFC", "untagged"]]) + list(code.iloc[1][["OB", "mPFC", "untagged"]]) + \
        [r.OB_mean, r.mPFC_mean, r.untagged_mean]
    within = [abs(v - e) <= t for v, e, t in zip(vals, exp_vals, tols)]
    print(r.mice, r.reading, "within tolerance:", within)

**Result: partially reproduced.** With the code's rule (planes without a red cell skipped), the identity dataset has 81 OB-p, 84 mPFC-p and 1937 untagged neurons from 10 mice, and the concentration dataset 87, 102 and 2132 from 10 mice, all within tolerance (over all planes the identity untagged count would be 2328). The Go/NoGo per-session mean of OB-p neurons is 103.3 (days 1 to 3) or 107.5 (days 0 to 3), outside 95 +/- 2 under every reading. The mPFC-p mean is within 87 +/- 8 only without sub-668 (91.7 and 94.5; 95.7 and 98.2 with it). The dandiset has 8 Go/NoGo mice, 4 of them mPFC-tagged, where the paper has 7.

## fast-dataset-counts

> The 30 Hz subset holds 24 OB-p, 27 mPFC-p and 424 untagged neurons from 5 mice.

In [ ]:
d = S[S.experiment_name == "fast_hz_sniff_trig_dataset"]
print(d[["subject", "projection_area", "n_cells", "n_red", "fs_timestamps"]].to_string(index=False))
print(counts(d, "n_cells_kept_planes"), "| paper: OB 24, mPFC 27, untagged 424, 5 mice")

**Result: reproduced.** 24 OB-p, 27 mPFC-p and 424 untagged neurons from 5 mice, exactly the paper's counts. The traces are sampled at 29.6 to 29.8 Hz.

## figs2b-labeling-similar

> OB-p and mPFC-p mice do not differ in the number of jGCaMP7f neurons or of co-labeled
> jGCaMP7f/tdTomato neurons.

Per mouse, suite2p cells (all planes) and red cells; unpaired two-tailed Student t-tests of OB-p
against mPFC-p mice. Readings: a mouse's count from (a) its first imaging session or (b) its
mean over sessions; the co-labeled measure as (c) a count or (d) a percentage of its jGCaMP7f
neurons.

In [ ]:
S["pct_red"] = 100 * S.n_red / S.n_cells
first = S.sort_values("date").groupby("subject").first()
mean = S.groupby("subject").agg(projection_area=("projection_area", "first"), n_cells=("n_cells", "mean"),
                                n_red=("n_red", "mean"), pct_red=("pct_red", "mean"))
res = []
for rname, df in [("a: first session", first), ("b: mean over sessions", mean)]:
    ob, mp = df[df.projection_area == "OB"], df[df.projection_area == "mPFC"]
    for col, lab in [("n_cells", "jGCaMP7f neurons"), ("n_red", "c: co-labeled count"), ("pct_red", "d: co-labeled %")]:
        tt = stats.ttest_ind(ob[col], mp[col])
        res.append(dict(reading=rname, measure=lab, n_OB=len(ob), n_mPFC=len(mp), OB_mean=ob[col].mean(),
                        mPFC_mean=mp[col].mean(), t=tt.statistic, p=tt.pvalue))
res = pd.DataFrame(res)
print(res.round(4).to_string(index=False))
print("paper: N = 9 OB-p and 10 mPFC-p mice, p = 0.555 (jGCaMP7f) and 0.455 (co-labeled)")
print("any p < 0.05:", bool((res.p < 0.05).any()))

**Result: reproduced.** No t-test reaches p < 0.05 under any reading: jGCaMP7f neurons p = 0.187 (first session) and 0.373 (mean over sessions), co-labeled count p = 0.585 and 0.985, co-labeled percentage p = 0.427 and 0.702. The dandiset has 10 OB-p and 8 mPFC-p mice, where the legend has 9 and 10.

## Identity dataset: binarized responses

The identity sessions are loaded and each trace normalized as the code does. Each cell's
binary activity on each trial is computed for the window 0 to 5 s after onset, under the two
readings of the window placement ("start", the main reading, and "centre"). Bad trials are left
out of everything; the baseline sample uses all good trials, odor and control. The 8 odors are
the non-control stimuli.

In [ ]:
ID = pcx.load_experiment(S, "identity_dataset")
for s in ID:
    tr = s["trials"]
    s["keep"] = (~tr.is_bad_trial.astype(bool) & ~tr.is_control).to_numpy()
    s["good"] = (~tr.is_bad_trial.astype(bool)).to_numpy()
rows = []
for s in ID:
    m, tr = s["meta"], s["trials"]
    rows.append(dict(subject=m["subject"], proj=m["projection_area"], cells=len(s["cells"]),
                     red=int(s["cells"].red.sum()), trials=len(tr), bad=int(tr.is_bad_trial.sum()),
                     odor_trials=int(s["keep"].sum()), odors=tr[s["keep"]].stim_name.nunique(),
                     min_trials_per_odor=int(tr[s["keep"]].stim_name.value_counts().min()),
                     fs=round(s["fs"], 3), onset_s=round(float(tr.stim_onset_timestamp.iloc[0] - tr.start_time.iloc[0]), 2),
                     X_min=float(s["X"].min()), X_max=float(s["X"].max())))
print(pd.DataFrame(rows).to_string(index=False))
print("odors:", sorted(set.intersection(*[set(s["trials"][s["keep"]].stim_name) for s in ID])))

In [ ]:
def binarize_sessions(sessions, win, starts_at_onset, seed, reading="start"):
    rng = np.random.default_rng(seed)
    return [pcx.binarize(s["X"], s["t"], s["fs"], win, starts_at_onset, rng, reading=reading,
                         trial_mask=s["good"]) for s in sessions]

B_ID = {rd: binarize_sessions(ID, (0, 5), True, 11, rd) for rd in ["start", "centre"]}
for rd, Bs in B_ID.items():
    print(rd, "mean active fraction on odor trials:",
          round(float(np.mean(np.concatenate([B[:, s["keep"]].ravel() for B, s in zip(Bs, ID)]))), 4))

def per_cell_table(sessions, Bs):
    out = []
    for s, B in zip(sessions, Bs):
        tr = s["trials"][s["keep"]]
        b = B[:, s["keep"]]
        odors = tr.stim_name.to_numpy()
        names = sorted(set(odors))
        r = np.stack([b[:, odors == o].mean(1) for o in names], 1)
        ls = pcx.vinje_gallant(r, 1)
        mi = np.array([pcx.mutual_info_bits(bi, odors) for bi in b])
        out.append(pd.DataFrame(dict(subject=s["meta"]["subject"], group=s["cells"].projection.values,
                                     lifetime_sparseness=ls, mutual_info=mi, mean_activity=b.mean(1))))
    return pd.concat(out, ignore_index=True)

CELL = {rd: per_cell_table(ID, B_ID[rd]) for rd in B_ID}
print(CELL["start"].groupby("group").agg(n=("subject", "size"), never_active=("mean_activity", lambda x: int((x == 0).sum()))).to_string())

GROUPS = ["OB", "mPFC", "untagged"]
PAIRS = [("OB", "mPFC"), ("OB", "untagged"), ("mPFC", "untagged")]

def pair_tests(values_by_group, m=3):
    rows = []
    for a, b in PAIRS:
        p = pcx.mwu(values_by_group[a], values_by_group[b])
        rows.append(dict(pair=f"{a} vs {b}", median_a=np.nanmedian(values_by_group[a]), median_b=np.nanmedian(values_by_group[b]),
                         mean_a=np.nanmean(values_by_group[a]), mean_b=np.nanmean(values_by_group[b]),
                         p=p, p_bonferroni=min(1.0, p * m)))
    return pd.DataFrame(rows)

## fig2b-lifetime-sparseness-similar

> Lifetime sparseness does not differ significantly between OB-p, mPFC-p and untagged neurons
> in the identity dataset.

Lifetime sparseness over the 8 odors of each cell's trial-averaged binary activity. A cell never
active on any odor trial has no defined sparseness and is left out (counts printed). Two-sided
Mann-Whitney U tests, Bonferroni over the 3 pairs.

In [ ]:
for rd, C in CELL.items():
    v = {g: C.loc[C.group == g, "lifetime_sparseness"].to_numpy() for g in GROUPS}
    print(f"reading {rd}: defined / total:", {g: f"{np.isfinite(v[g]).sum()}/{len(v[g])}" for g in GROUPS})
    print(pair_tests(v).round(4).to_string(index=False))
    if rd == "start":
        # by animal
        print(C.groupby(["group", "subject"]).lifetime_sparseness.median().unstack(0).round(3).to_string())

**Result: not reproduced.** mPFC-p and untagged neurons differ (medians 0.774 and 0.714, Bonferroni p = 0.0067 under the start reading and 0.015 under the centre reading). OB-p against mPFC-p (corrected p = 1.0) and OB-p against untagged (0.38 and 0.42) do not differ. Under the start reading 21 OB-p, 11 mPFC-p and 230 untagged neurons are never active on an odor trial; they have no sparseness and are left out.

## fig2c-mutual-information

> Mutual information between binarized activity and odor identity does not differ between OB-p
> and mPFC-p neurons, but is lower for OB-p than untagged neurons (p = 0.012).

In [ ]:
for rd, C in CELL.items():
    v = {g: C.loc[C.group == g, "mutual_info"].to_numpy() for g in GROUPS}
    print(f"reading {rd}")
    print(pair_tests(v).round(5).to_string(index=False))
print(CELL["start"].groupby(["group", "subject"]).mutual_info.median().unstack(0).round(4).to_string())

**Result: not reproduced.** OB-p and mPFC-p do not differ (corrected p = 1.0), as the paper says, but OB-p is not significantly lower than untagged: medians 0.069 and 0.079 bits, corrected p = 0.19 (start) and 0.17 (centre).

## Pseudo-population trials

The identity pseudo-population: cells of all mice of a group, trials matched by odor and by the
order of that odor's presentation, truncated to the smallest count across mice.

In [ ]:
def group_masks(sessions, g):
    return [(s["cells"].projection == g).to_numpy() for s in sessions]

def pseudo(sessions, g, arrays=None, key="stim_name"):
    return pcx.pseudo_population(sessions, group_masks(sessions, g), key, [s["keep"] for s in sessions],
                                 arrays=arrays)

PB = {}
for rd in B_ID:
    for g in GROUPS:
        tab, Pb = pseudo(ID, g, arrays=B_ID[rd])
        PB[rd, g] = Pb
print("pseudo-population trials:", len(tab), "| per odor:", tab.stim_name.value_counts().to_dict())
print("cells per group:", {g: PB["start", g].shape[0] for g in GROUPS})

## fig2d-population-sparseness

> Population sparseness per trial differs only between OB-p and untagged populations
> (p = 1.26 x 10^-5).

Per pseudo-population trial, population sparseness over all cells of the group from their binary
activity on that trial. With binary values this equals (1 - q) / (1 - 1/N) for a fraction q of
active cells.

In [ ]:
for rd in B_ID:
    v = {g: pcx.vinje_gallant(PB[rd, g].T.astype(float), 1) for g in GROUPS}
    print(f"reading {rd}: undefined trials (no cell active):", {g: int(np.isnan(v[g]).sum()) for g in GROUPS})
    t2d = pair_tests(v)
    print(t2d.round(7).to_string(index=False))
    print("significant pairs (corrected p < 0.05):", list(t2d.pair[t2d.p_bonferroni < 0.05]))

**Result: not reproduced.** All three pairs differ, not only OB-p against untagged: OB-p against mPFC-p corrected p = 8.7e-06 (start) and 4.7e-05 (centre), OB-p against untagged 1e-07 and 2e-06, mPFC-p against untagged below 1e-07. OB-p population sparseness (median 0.925) lies between untagged (0.886) and mPFC-p (0.952).

## fig2-proportion-responding

> A higher proportion of OB-p than mPFC-p neurons responds to odorants.

Readings: (a) per pseudo-population trial, the fraction of the group's cells active (64 values
per group); (b) per neuron, the fraction of its odor trials on which it is active.

In [ ]:
for rd in B_ID:
    va = {g: PB[rd, g].mean(0) for g in GROUPS}
    C = CELL[rd]
    vb = {g: C.loc[C.group == g, "mean_activity"].to_numpy() for g in GROUPS}
    print(f"reading {rd}, (a) per pseudo-population trial")
    print(pair_tests(va).round(8).to_string(index=False))
    print(f"reading {rd}, (b) per neuron")
    print(pair_tests(vb).round(8).to_string(index=False))
print("per mouse, mean fraction active on odor trials (start reading):")
print(CELL["start"].groupby(["group", "subject"]).mean_activity.mean().unstack(0).round(3).to_string())

**Result: reproduced.** Under reading (a), per pseudo-population trial, 9.49% of OB-p and 6.94% of mPFC-p neurons are active (medians 8.6% and 6.0%), corrected p = 9e-06 (start) and 4.7e-05 (centre). Under reading (b), per neuron, the difference is not significant (corrected p = 1.0).

## fig2e-reliability

> OB-p populations contain higher fractions of both unreliable and reliable responsive cells per
> trial (10.97% and 1.54%) than mPFC-p populations (8.71% and 0.25%).

Methods, Reliability analysis: a cell responds on a trial when its mean normalized fluorescence
0 to 5 s after onset exceeds its baseline mean plus 2.33 baseline SDs; reliable when it responds
on more than half of its trials. Readings: baseline (a) the 4 s before onset or (b) the 5 s
before; mean and SD (c) per trial or (d) pooled over the cell's odor trials; "more than half"
over (e) all odor trials or (f) the trials of any one odor; samples (g) pseudo-population trials
or (h) each mouse's own odor trials, pooled by group.

In [ ]:
def reliability(s, base, per_trial, rule):
    X, t = s["X"][:, s["keep"]], s["t"]
    bm = (t >= -base) & (t < 0)
    resp = X[:, :, (t >= 0) & (t < 5)].mean(-1)
    if per_trial:
        mu, sd = X[:, :, bm].mean(-1), X[:, :, bm].std(-1, ddof=1)
    else:
        B = X[:, :, bm].reshape(X.shape[0], -1)
        mu, sd = B.mean(1)[:, None], B.std(1, ddof=1)[:, None]
    r = resp > mu + 2.33 * sd
    odors = s["trials"][s["keep"]].stim_name.to_numpy()
    if rule == "e":
        rel = r.mean(1) > 0.5
    else:
        rel = np.stack([r[:, odors == o].mean(1) > 0.5 for o in np.unique(odors)], 1).any(1)
    full = np.zeros((X.shape[0], len(s["trials"])), np.int8)
    full[:, s["keep"]] = r * (1 + rel[:, None])         # 0 none, 1 responsive unreliable, 2 responsive reliable
    return full

rel_rows = []
for base, bl in [(4, "a"), (5, "b")]:
    for per_trial, pl in [(True, "c"), (False, "d")]:
        for rule in ["e", "f"]:
            R = [reliability(s, base, per_trial, rule) for s in ID]
            for samp in ["g", "h"]:
                vals = {}
                for g in ["OB", "mPFC"]:
                    if samp == "g":
                        _, P = pseudo(ID, g, arrays=R)
                    else:
                        P = None
                    out = {}
                    for kind, code in [("unreliable", 1), ("reliable", 2)]:
                        if samp == "g":
                            out[kind] = 100 * (P == code).mean(0)
                        else:
                            out[kind] = np.concatenate([100 * (Ri[(s["cells"].projection == g).to_numpy()][:, s["keep"]] == code).mean(0)
                                                        for Ri, s in zip(R, ID) if (s["cells"].projection == g).any()])
                    vals[g] = out
                pu = pcx.mwu(vals["OB"]["unreliable"], vals["mPFC"]["unreliable"])
                pr = pcx.mwu(vals["OB"]["reliable"], vals["mPFC"]["reliable"])
                rel_rows.append(dict(reading=bl + pl + rule + samp,
                                     OB_unrel=vals["OB"]["unreliable"].mean(), OB_rel=vals["OB"]["reliable"].mean(),
                                     mPFC_unrel=vals["mPFC"]["unreliable"].mean(), mPFC_rel=vals["mPFC"]["reliable"].mean(),
                                     p_unrel=pu, p_rel=pr, p_unrel_bonf=min(1, 3 * pu), p_rel_bonf=min(1, 3 * pr), n_trials=len(vals["OB"]["reliable"])))
REL = pd.DataFrame(rel_rows)
exp4, tol4 = np.array([10.97, 1.54, 8.71, 0.25]), np.array([1.097, 0.154, 0.871, 0.025])
REL["values_within"] = REL.apply(lambda r: int((np.abs(np.array([r.OB_unrel, r.OB_rel, r.mPFC_unrel, r.mPFC_rel]) - exp4) <= tol4).sum()), axis=1)
REL["tests_hold"] = (REL.p_unrel_bonf < 0.05) & (REL.p_rel_bonf < 0.05) & (REL.OB_unrel > REL.mPFC_unrel) & (REL.OB_rel > REL.mPFC_rel)
# the correction count is not stated for this panel: Bonferroni over the 2 tests is shown as a second reading
REL["tests_hold_m2"] = (2 * REL.p_unrel < 0.05) & (2 * REL.p_rel < 0.05) & (REL.OB_unrel > REL.mPFC_unrel) & (REL.OB_rel > REL.mPFC_rel)
print("Bonferroni over the 3 group pairs, as for the other Figure 2 comparisons; values_within counts the 4 percentages within tolerance")
print(REL.round(4).to_string(index=False))
print("combinations where both tests hold, Bonferroni over 3 pairs:", list(REL.reading[REL.tests_hold]),
      "| over 2 tests:", list(REL.reading[REL.tests_hold_m2]))

**Result: not reproduced, resting on an assumption.** With Bonferroni over the 3 group pairs, as for the other Figure 2 comparisons, both tests hold under no combination of readings. OB-p has more reliable responsive cells per trial than mPFC-p under all 16 combinations (corrected p <= 0.0003), but not more unreliable ones (smallest corrected p = 0.063, under b, c, e, h). With Bonferroni over the 2 tests, both tests hold under b, c, e, h (unreliable 17.23% against 15.93%, raw p = 0.0211), which would make the claim partially reproduced. At most one of the four percentages is within tolerance under any combination; for example under a, c, f, g OB-p has 7.64% unreliable and 16.20% reliable cells against 8.69% and 12.02% for mPFC-p.

## Identity decoding

Pseudo-populations of normalized fluorescence, matched as above; a linear SVM
(`SVC(kernel='linear')`) decodes the 8 odors from each cell's mean in a window, with 5-fold
stratified cross-validation; each repeat draws 75 cells of the group without replacement and a
new fold split, and its accuracy is the mean over folds.

In [ ]:
PX = {g: pseudo(ID, g)[1] for g in GROUPS}
tabX, _ = pseudo(ID, "OB")
y_id = tabX.stim_name.to_numpy()
# the sessions share the onset-aligned frame grid (same frame count before and after onset)
print("frames per trial:", {len(s["t"]) for s in ID}, "| pre-onset frames:", {s["n_pre"] for s in ID})
t_id = ID[0]["t"]
print({g: PX[g].shape for g in GROUPS}, "| chance 12.5%")

## fig2h-identity-decoding-5s

> Odor identity decoded from 5 s windows reaches similar accuracy in size-matched OB-p and mPFC-p
> pseudo-populations, above chance and slightly below untagged.

200 repeats of 75 cells, windows (a) 0 to 5 s (legend) and (b) 1 to 5 s (Methods).

In [ ]:
rng_h = np.random.default_rng(2008)
ACC5 = {}
for wname, win in [("a: 0-5 s", (0, 5)), ("b: 1-5 s", (1, 5))]:
    for g in GROUPS:
        ACC5[wname, g] = pcx.decode_windows(PX[g], y_id, t_id, [win], 75, 200, rng_h)[:, 0]
    print(wname, {g: round(100 * ACC5[wname, g].mean(), 2) for g in GROUPS},
          "| OB vs mPFC p =", f"{pcx.mwu(ACC5[wname, 'OB'], ACC5[wname, 'mPFC']):.4g}",
          "| OB vs untagged p =", f"{pcx.mwu(ACC5[wname, 'OB'], ACC5[wname, 'untagged']):.3g}",
          "| mPFC vs untagged p =", f"{pcx.mwu(ACC5[wname, 'mPFC'], ACC5[wname, 'untagged']):.3g}")
    for g in GROUPS:
        tt = stats.ttest_1samp(ACC5[wname, g], 0.125, alternative="greater")
        print(f"   {g}: min {ACC5[wname, g].min():.3f}, max {ACC5[wname, g].max():.3f}, one-sided t-test against 12.5% p = {tt.pvalue:.3g}")

**Result: blocked (methods underspecified).** The two windows disagree. With 0 to 5 s (legend) OB-p and mPFC-p reach 25.7% and 25.6%, p = 0.675, no difference; with 1 to 5 s (Methods) 28.5% and 31.1%, p = 9.7e-09. Under both, every group is above 12.5% (p < 1e-100) and below untagged (41.9% and 49.0%). The criterion blocks the claim when the windows disagree.

## fig2i-identity-decoding-early

> In an early window (0 to 2 s) OB-p pseudo-populations decode odor identity better than mPFC-p
> pseudo-populations.

In [ ]:
rng_i = np.random.default_rng(2009)
ACC2 = {g: pcx.decode_windows(PX[g], y_id, t_id, [(0, 2)], 75, 200, rng_i)[:, 0] for g in GROUPS}
print({g: round(100 * ACC2[g].mean(), 2) for g in GROUPS},
      "| OB vs mPFC two-sided MWU p =", f"{pcx.mwu(ACC2['OB'], ACC2['mPFC']):.3g}")
print({g: (round(ACC2[g].min(), 3), round(ACC2[g].max(), 3)) for g in GROUPS})

**Result: reproduced.** In 0 to 2 s OB-p reaches 17.76% and mPFC-p 16.33% (untagged 25.09%), two-sided Mann-Whitney p = 5.0e-05 over the 200 repeats. The difference is 1.4 points.

## fig2j-identity-decoding-peaks-earlier

> With a sliding window, odor identity decoding accuracy peaks earlier for OB-p than for mPFC-p
> pseudo-populations.

Non-overlapping windows from 5 s before to 15 s after onset, (a) 250 ms and (b) 500 ms wide,
trained and tested on the same window, 75 cells, 100 repeats. At 4.51 Hz a 250 ms window holds
one frame (sometimes two, sometimes none; empty windows are dropped, count printed). The peak
time is the start of the window with the highest mean accuracy; its 95% interval is that of the
per-repeat peak times.

In [ ]:
def windows_of(width, t, a=-5.0, b=15.0):
    edges = np.arange(a, b + 1e-9, width)
    wins = [(lo, lo + width) for lo in edges[:-1]]
    keep = [w for w in wins if ((t >= w[0]) & (t < w[1])).any()]
    return keep, len(wins) - len(keep)

SLIDE = {}
for wi, (wname, width) in enumerate([("a: 250 ms", 0.25), ("b: 500 ms", 0.5)]):
    wins, n_empty = windows_of(width, t_id)
    starts = np.array([w[0] for w in wins])
    print(f"{wname}: {len(wins)} windows, {n_empty} empty windows dropped")
    # each group has its own generator (2010 + 10 * window reading + group index); groups run in parallel
    res = Parallel(n_jobs=3)(delayed(pcx.decode_windows)(PX[g], y_id, t_id, wins, 75, 100, np.random.default_rng(2010 + 10 * wi + i))
                             for i, g in enumerate(GROUPS))
    for g, A in zip(GROUPS, res):
        SLIDE[wname, g] = (starts, A)
    for g in GROUPS:
        A = SLIDE[wname, g][1]
        m = A.mean(0)
        pk = starts[np.argmax(m)]
        per = starts[np.argmax(A, 1)]
        print(f"   {g}: peak mean accuracy {100 * m.max():.2f}% at {pk:.2f} s; per-repeat peak times 95% interval "
              f"{np.percentile(per, 2.5):.2f} to {np.percentile(per, 97.5):.2f} s; median {np.median(per):.2f} s")
    print("   OB peak earlier than mPFC:", SLIDE[wname, "OB"][0][np.argmax(SLIDE[wname, "OB"][1].mean(0))] <
          SLIDE[wname, "mPFC"][0][np.argmax(SLIDE[wname, "mPFC"][1].mean(0))])

print("clock check (identity decoding before and after onset, 500 ms windows):")
st, _ = SLIDE["b: 500 ms", "OB"]
for g in GROUPS:
    A = SLIDE["b: 500 ms", g][1]
    print(f"   {g}: mean accuracy over windows starting -5 to 0 s {100 * A[:, st < 0].mean():.2f}%, 0 to 2 s {100 * A[:, (st >= 0) & (st < 2)].mean():.2f}% (chance 12.5%)")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, wname in zip(axs, ["a: 250 ms", "b: 500 ms"]):
    for g, c in zip(GROUPS, ["tab:cyan", "tab:pink", "gray"]):
        st, A = SLIDE[wname, g]
        ax.plot(st, 100 * A.mean(0), color=c, label=g)
    ax.axhline(12.5, ls="--", color="k"); ax.set_title(wname); ax.set_xlabel("window start (s from onset)")
    ax.set_ylabel("accuracy (%)"); ax.legend()
plt.show()
print("plotted mean accuracies range:", {k: (round(100 * v[1].mean(0).min(), 1), round(100 * v[1].mean(0).max(), 1)) for k, v in SLIDE.items()})

**Result: reproduced.** The mean curve peaks at 5.75 s for OB-p and 9.25 s for mPFC-p (250 ms windows), and at 5.50 and 9.00 s (500 ms windows). The per-repeat peak times overlap widely (250 ms: OB-p 3.87 to 7.00 s, mPFC-p 2.12 to 12.17 s; medians 5.75 and 7.62 s), so the mPFC-p peak time is poorly determined. Untagged peaks earliest (2.50 and 3.50 s). The clock check holds: before onset all groups are at or below chance (9.3 to 12.0%), and 0 to 2 s after onset they are above it (18.8 to 26.2%).

## fig2k-mpfc-late-identity

> Late in the trial, mPFC-p pseudo-populations keep low but above-chance identity decoding that
> exceeds OB-p decoding.

From the same-window accuracies above: per repeat, the mean over windows starting within
(a) 2 to 15 s or (b) 5 to 15 s after onset. One-sample one-sided t-test of mPFC-p against 12.5%
and two-sided Mann-Whitney U of mPFC-p against OB-p.

In [ ]:
for wname in SLIDE.keys() if False else ["a: 250 ms", "b: 500 ms"]:
    for lname, (lo, hi) in [("a: 2-15 s", (2, 15)), ("b: 5-15 s", (5, 15))]:
        st = SLIDE[wname, "OB"][0]
        sel = (st >= lo) & (st < hi)
        mp = SLIDE[wname, "mPFC"][1][:, sel].mean(1)
        ob = SLIDE[wname, "OB"][1][:, sel].mean(1)
        un = SLIDE[wname, "untagged"][1][:, sel].mean(1)
        print(f"window {wname}, late {lname}: mPFC {100 * mp.mean():.2f}%, OB {100 * ob.mean():.2f}%, untagged {100 * un.mean():.2f}%; "
              f"mPFC > 12.5% p = {stats.ttest_1samp(mp, 0.125, alternative='greater').pvalue:.3g}; "
              f"mPFC vs OB p = {pcx.mwu(mp, ob):.3g}")

**Result: reproduced.** Late in the trial mPFC-p reaches 31.9 to 33.0% under the four combinations of readings, above 12.5% (p <= 1.7e-88) and above OB-p (30.8 to 31.0%, p <= 3.8e-05). The differences are 1.2 to 2.1 points, and untagged is higher than both (38.8 to 41.7%).

## 30 Hz dataset: calcium peaks

Peaks are found per trial on normalized fluorescence with `scipy.signal.find_peaks`, using the
Methods' minimum criteria as its arguments: width 100 ms, distance 1 s, height 0.5, prominence
0.25. The first second of each trial is left out (the trace starts 4 s before onset; trials start
about 5.1 s before onset), and the trace ends 15 s after onset, more than a second before each
trial's end. Odor trials only (bad and control trials left out).

In [ ]:
FAST = pcx.load_experiment(S, "fast_hz_sniff_trig_dataset")
print([(s["meta"]["subject"], s["meta"]["projection_area"], s["X"].shape, round(s["fs"], 2),
        int(s["trials"].is_bad_trial.sum())) for s in FAST])
peak_rows = []
for s in FAST:
    tr = s["trials"]
    keep = (~tr.is_bad_trial.astype(bool) & ~tr.is_control).to_numpy()
    fs, t = s["fs"], s["t"]
    seg = t >= -4.0
    ts = t[seg]
    for c in range(s["X"].shape[0]):
        for k in np.where(keep)[0]:
            pk, _ = signal.find_peaks(s["X"][c, k, seg], width=0.1 * fs, distance=1.0 * fs, height=0.5, prominence=0.25)
            peak_rows.append(dict(subject=s["meta"]["subject"], cell=c, trial=k,
                                  group=s["cells"].projection.iloc[c], peaks=ts[pk]))
PK = pd.DataFrame(peak_rows)
PK["first_after"] = PK.peaks.apply(lambda p: p[p >= 0][0] if (p >= 0).any() else np.nan)
print("cell-trials:", len(PK), "| with a peak after onset:", int(PK.first_after.notna().sum()))
print(PK.groupby("group").agg(cells=("cell", lambda x: 0), cell_trials=("trial", "size"),
                              with_peak=("first_after", lambda x: int(x.notna().sum()))).drop(columns="cells").to_string())

## fig2f-latency

> In the 30 Hz data OB-p neurons reach their first calcium peak after odor onset sooner than
> mPFC-p neurons; OB-p and untagged neurons do not differ.

Latency of the first peak after onset, (a) per trial (every trial with a peak) and (b) per neuron
(median over its trials with a peak). Two-sided Mann-Whitney U, Bonferroni over the 3 pairs.

In [ ]:
lat_a = {g: PK.loc[PK.group == g, "first_after"].dropna().to_numpy() for g in GROUPS}
nb = PK.dropna(subset=["first_after"]).groupby(["group", "subject", "cell"]).first_after.median().reset_index()
lat_b = {g: nb.loc[nb.group == g, "first_after"].to_numpy() for g in GROUPS}
for name, v in [("a: per trial", lat_a), ("b: per neuron", lat_b)]:
    print(name, {g: len(v[g]) for g in GROUPS})
    print(pair_tests(v).to_string(index=False))
print("per mouse median latency per trial (s):")
print(PK.dropna(subset=["first_after"]).groupby(["group", "subject"]).first_after.median().round(2).to_string())

**Result: reproduced.** (a) Per trial, the median latency is 3.58 s for OB-p and 4.65 s for mPFC-p, corrected p = 3.1e-16; OB-p against untagged (3.61 s) has corrected p = 1.0. (b) Per neuron, 3.84 against 4.62 s, corrected p = 0.030, and OB-p against untagged p = 1.0. By mouse, the OB-p medians are 2.70 to 4.96 s and the mPFC-p medians 4.59 to 4.79 s.

## fig2g-early-peaks

> OB-p neurons have more calcium peaks at early time points after odor onset than mPFC-p
> neurons.

Peaks per trial in (a) 0 to 2 s and (b) 0 to 3 s after onset, averaged per neuron. The
two-sided Mann-Whitney U test of the per-neuron means is shown as a reference; the claim is
judged on the means.

In [ ]:
for name, (lo, hi) in [("a: 0-2 s", (0, 2)), ("b: 0-3 s", (0, 3))]:
    PK["n_early"] = PK.peaks.apply(lambda p: int(((p >= lo) & (p < hi)).sum()))
    per = PK.groupby(["group", "subject", "cell"]).n_early.mean().reset_index()
    v = {g: per.loc[per.group == g, "n_early"].to_numpy() for g in GROUPS}
    print(name, "mean peaks per trial per neuron:", {g: round(v[g].mean(), 4) for g in GROUPS},
          "| OB vs mPFC p =", f"{pcx.mwu(v['OB'], v['mPFC']):.3g}")
    print("   per mouse:", per.groupby(["group", "subject"]).n_early.mean().round(3).to_dict())
bins = np.arange(-4, 15.01, 0.5)
hist = {g: np.histogram(np.concatenate(PK.loc[PK.group == g, "peaks"].to_list()), bins)[0] /
        PK.loc[PK.group == g].shape[0] for g in GROUPS}
fig, ax = plt.subplots(figsize=(6, 3))
for g, c in zip(GROUPS, ["tab:cyan", "tab:pink", "gray"]):
    ax.plot(bins[:-1], hist[g], color=c, label=g)
ax.set_xlabel("time from onset (s), 0.5 s bins"); ax.set_ylabel("peaks per trial"); ax.legend(); plt.show()
print("plotted range:", {g: (round(h.min(), 4), round(h.max(), 4)) for g, h in hist.items()})

**Result: reproduced.** Mean peaks per trial per neuron, 0 to 2 s: OB-p 0.198, mPFC-p 0.138 (Mann-Whitney p = 0.17); 0 to 3 s: 0.349 and 0.203 (p = 0.026). The difference rests on two of the three OB-p mice: sub-2401 has fewer early peaks (0.073 and 0.122) than either mPFC-p mouse.

## Concentration dataset

Three odorants at three concentrations (stored as fractions 0.001, 0.01 and 0.1, the 0.1%, 1%
and 10% of the Figure 3 and S5 legends) and a control.

In [ ]:
CO = pcx.load_experiment(S, "concentration_dataset")
for s in CO:
    tr = s["trials"]
    s["keep"] = (~tr.is_bad_trial.astype(bool) & ~tr.is_control).to_numpy()
    s["good"] = (~tr.is_bad_trial.astype(bool)).to_numpy()
    tr["stim"] = tr.stim_name + " " + tr.conc.astype(str)
print(pd.DataFrame([dict(subject=s["meta"]["subject"], proj=s["meta"]["projection_area"], cells=len(s["cells"]),
                         red=int(s["cells"].red.sum()), bad=int(s["trials"].is_bad_trial.sum()),
                         stimuli=s["trials"][s["keep"]].stim.nunique(),
                         min_trials=int(s["trials"][s["keep"]].stim.value_counts().min()),
                         fs=round(s["fs"], 3)) for s in CO]).to_string(index=False))
print("concentrations:", sorted(CO[0]["trials"].conc.dropna().unique()))
for s in CO:
    tr = s["trials"][s["keep"]]
    rho = stats.spearmanr(tr.index, tr.conc)
    print(f"   sub-{s['meta']['subject']}: Spearman correlation of trial number with concentration {rho.statistic:+.2f} (p = {rho.pvalue:.2g}); "
          f"mean trial number at 0.1%, 1%, 10%: {tr.groupby('conc').apply(lambda d: d.index.to_series().mean()).round(1).tolist()}")

## fig3bc-concentration-tuning

> Similar fractions of OB-p and mPFC-p neurons are tuned to odor identity, but OB-p neurons more
> often show significant effects of concentration and identity by concentration interactions,
> which are rare in mPFC-p neurons.

Per neuron, a two-way ANOVA (identity by concentration, with interaction; type II sums of
squares, the same as any type for a balanced design) on mean normalized fluorescence 0 to 5 s
after onset, p < 0.05 per term. Time-resolved version in non-overlapping windows of (a) 500 ms
and (b) 1 s.

In [ ]:
def anova2(Y, A, B):
    """Type II two-way ANOVA with interaction for every column of Y (trials x cells)."""
    a = pd.get_dummies(pd.Categorical(A), drop_first=True).to_numpy(float)
    b = pd.get_dummies(pd.Categorical(B), drop_first=True).to_numpy(float)
    ab = np.einsum("ti,tj->tij", a, b).reshape(len(A), -1)
    one = np.ones((len(A), 1))
    def rss(M):
        beta, *_ = np.linalg.lstsq(M, Y, rcond=None)
        return ((Y - M @ beta) ** 2).sum(0)
    r_a, r_b = rss(np.hstack([one, a])), rss(np.hstack([one, b]))
    r_ab = rss(np.hstack([one, a, b]))
    full = np.hstack([one, a, b, ab])
    r_full = rss(full)
    df_res = len(A) - np.linalg.matrix_rank(full)
    out = {}
    for name, ss, df in [("identity", r_b - r_ab, a.shape[1]), ("concentration", r_a - r_ab, b.shape[1]),
                         ("interaction", r_ab - r_full, ab.shape[1])]:
        F = (ss / df) / (r_full / df_res)
        out[name] = stats.f.sf(F, df, df_res)
    return out

def tuning_fractions(win):
    rows = []
    for s in CO:
        tr = s["trials"][s["keep"]]
        Y = pcx.window_mean(s["X"][:, s["keep"]], s["t"], *win).T
        P = anova2(Y, tr.stim_name.to_numpy(), tr.conc.to_numpy())
        rows.append(pd.DataFrame(dict(group=s["cells"].projection.values, **{k: v < 0.05 for k, v in P.items()})))
    D = pd.concat(rows)
    return 100 * D.groupby("group")[["identity", "concentration", "interaction"]].mean()

F05 = tuning_fractions((0, 5))
print("percent of neurons with p < 0.05, 0 to 5 s:")
print(F05.round(2).to_string())
d_id = abs(F05.loc["OB", "identity"] - F05.loc["mPFC", "identity"])
d_conc = abs(F05.loc["OB", "concentration"] - F05.loc["mPFC", "concentration"])
print(f"OB > mPFC for concentration: {F05.loc['OB', 'concentration'] > F05.loc['mPFC', 'concentration']}; "
      f"for interaction: {F05.loc['OB', 'interaction'] > F05.loc['mPFC', 'interaction']}; "
      f"|OB - mPFC| identity {d_id:.2f} < concentration {d_conc:.2f}: {d_id < d_conc}")
for wname, width in [("a: 500 ms", 0.5), ("b: 1 s", 1.0)]:
    edges = np.arange(-2, 15 + 1e-9, width)
    tc = {lo: tuning_fractions((lo, lo + width)) for lo in edges[:-1]}
    tcd = pd.concat(tc, names=["start"]).unstack("group")
    print(f"\n{wname}: time course of percent significant (rows: window start, s)")
    print(tcd.round(1).to_string())
    post = tcd[tcd.index >= 0]
    print("mean over windows 0 to 15 s:", post.mean().round(2).to_dict())

**Result: reproduced.** In 0 to 5 s, 17.2% of OB-p and 15.7% of mPFC-p neurons are tuned to identity, 10.3% and 6.9% to concentration, and 9.2% and 6.9% show an interaction. OB-p exceeds mPFC-p for concentration and interaction, and the identity difference (1.6 points) is smaller than the concentration difference (3.5 points). Over the time course OB-p is higher for concentration mainly 1.5 to 4.5 s after onset (11.5 to 16.1% against 4.9 to 9.8% in 500 ms windows), while the means over 0 to 15 s are close (10.3 and 9.5%; 9.8 and 9.8% in 1 s windows). The mPFC-p interaction fraction (6.9%) is near the 5% expected by chance. Untagged neurons have the highest fractions on all three terms (22.2, 11.3 and 10.5%).

## fig3de-concentration-decoding

> OB-p pseudo-populations decode odor concentration more accurately and earlier than mPFC-p and
> untagged pseudo-populations, while mPFC-p decoding stays close to chance.

Per odor, a linear SVM decodes the 3 concentrations in non-overlapping windows of (a) 250 ms and
(b) 500 ms from 2 s before to 15 s after onset, trained and tested on the same window, 75 cells,
100 repeats; accuracies averaged over the 3 odors; chance 33.3%. Pseudo-population trials are
matched by odor and concentration. The same repeats and folds give the pairwise decoders of
figs5g-pairwise-concentration.

In [ ]:
PC = {}
for g in GROUPS:
    tab_c, P = pcx.pseudo_population(CO, group_masks(CO, g), "stim", [s["keep"] for s in CO])
    PC[g] = P
odor_c = tab_c.stim.str.rsplit(" ", n=1).str[0].to_numpy()
conc_c = tab_c.stim.str.rsplit(" ", n=1).str[1].astype(float).to_numpy()
conc_code = np.round(np.log10(conc_c)).astype(int)   # -3, -2, -1 for 0.1%, 1%, 10%
t_co = CO[0]["t"]
print("pseudo-population trials:", len(tab_c), tab_c.stim.value_counts().min(), "to", tab_c.stim.value_counts().max(), "per stimulus;",
      {g: PC[g].shape[0] for g in GROUPS}, "cells")

def conc_decoding_group(P, width, n_rep, seed):
    wins, _ = windows_of(width, t_co, -2.0, 15.0)
    rng = np.random.default_rng(seed)
    three = np.zeros((n_rep, len(wins)))
    pairs = {pr: np.zeros((n_rep, len(wins))) for pr in [(-3, -2), (-2, -1), (-3, -1)]}
    for o in np.unique(odor_c):
        sel = odor_c == o
        three += pcx.decode_windows(P[:, sel], conc_code[sel], t_co, wins, 75, n_rep, rng) / 3
        for pr in pairs:
            sel2 = sel & np.isin(conc_code, pr)
            pairs[pr] += pcx.decode_windows(P[:, sel2], conc_code[sel2], t_co, wins, 75, n_rep, rng) / 3
    return np.array([w[0] for w in wins]), dict(three=three, pairs=pairs)

def conc_decoding(width, n_rep, seed):
    """Each group has its own generator (seed + group index); the groups run in parallel."""
    res = Parallel(n_jobs=3)(delayed(conc_decoding_group)(PC[g], width, n_rep, seed + i) for i, g in enumerate(GROUPS))
    return res[0][0], {g: r[1] for g, r in zip(GROUPS, res)}

CD = {wname: conc_decoding(width, 100, seed) for wname, width, seed in [("a: 250 ms", 0.25, 3001), ("b: 500 ms", 0.5, 3002)]}

In [ ]:
for wname, (st, out) in CD.items():
    pk = {g: (100 * out[g]["three"].mean(0).max(), st[np.argmax(out[g]["three"].mean(0))]) for g in GROUPS}
    print(wname, "peak mean accuracy (%) and its window start (s):", {g: (round(a, 2), round(b, 2)) for g, (a, b) in pk.items()})
    for g in GROUPS:
        per = st[np.argmax(out[g]["three"], 1)]
        print(f"   {g}: per-repeat peak time 95% interval {np.percentile(per, 2.5):.2f} to {np.percentile(per, 97.5):.2f} s; "
              f"pre-onset mean accuracy {100 * out[g]['three'][:, st < 0].mean():.2f}%")
    mid = (100 / 3 + pk["OB"][0]) / 2
    print(f"   OB peak > mPFC and untagged: {pk['OB'][0] > pk['mPFC'][0] and pk['OB'][0] > pk['untagged'][0]}; "
          f"OB earlier than both: {pk['OB'][1] < pk['mPFC'][1] and pk['OB'][1] < pk['untagged'][1]}; "
          f"mPFC peak {pk['mPFC'][0]:.2f} below midpoint {mid:.2f}: {pk['mPFC'][0] < mid}")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, (wname, (st, out)) in zip(axs, CD.items()):
    for g, c in zip(GROUPS, ["tab:cyan", "tab:pink", "gray"]):
        ax.plot(st, 100 * out[g]["three"].mean(0), color=c, label=g)
    ax.axhline(100 / 3, ls="--", color="k"); ax.set_title(wname); ax.set_xlabel("window start (s)"); ax.legend()
plt.show()

**Result: partially reproduced.** OB-p peaks highest (61.9% with 250 ms windows, 62.0% with 500 ms) and earliest (3.75 and 3.50 s), ahead of untagged (55.7% at 4.75 s; 54.0% at 4.50 s) and mPFC-p (53.6% at 9.75 s; 52.4% at 9.50 s). mPFC-p is not close to chance by the criterion's bound: its peak exceeds the midpoint between chance and the OB-p peak (47.6% under both readings). Before onset mPFC-p already decodes concentration at 37.2 to 37.6% (chance 33.3%), and trial order is not correlated with concentration in any session (|Spearman rho| <= 0.02), so that offset is not explained by the presentation order.

## figs5g-pairwise-concentration

> Pairwise concentration decoding (low against medium, medium against high, low against high) is
> strongest and most sustained in OB-p ensembles, intermediate in untagged neurons and weakest in
> mPFC-p neurons.

Binary SVMs for each pair of concentrations per odor (chance 50%), averaged over odors; the mean
same-window accuracy over windows starting 0 to 15 s after onset, per group.

In [ ]:
for wname, (st, out) in CD.items():
    sel = st >= 0
    for pr, lab in [((-3, -2), "0.1% vs 1%"), ((-2, -1), "1% vs 10%"), ((-3, -1), "0.1% vs 10%")]:
        m = {g: 100 * out[g]["pairs"][pr][:, sel].mean() for g in GROUPS}
        print(f"{wname} {lab}: " + ", ".join(f"{g} {m[g]:.2f}%" for g in GROUPS) +
              f" | order OB > untagged > mPFC: {m['OB'] > m['untagged'] > m['mPFC']}")

**Result: not reproduced.** The order OB-p > untagged > mPFC-p holds only for 0.1% against 1% (OB-p 62.3 and 62.4%, untagged 62.1 and 61.4%, mPFC-p 56.2 and 56.7%). For 1% against 10% and for 0.1% against 10% the three groups are within 2.2 points of each other and untagged is highest under both window readings.

## figs5-sparseness-concentration

> Across the three concentrations, response sparseness and the fraction of active cells change
> only modestly in all three populations.

Binarized activity 0 to 5 s ("start" reading). Per concentration: the percentage of active cells
per trial (over each group's cells, each mouse's trials), lifetime sparseness per neuron over the
3 odors at that concentration (mean over neurons with a defined value), and population sparseness
per subject (over the subject's cells of the group, per trial, averaged over trials). "Only
modestly" is judged as no population showing a monotonic change of more than half its value at
0.1% across the range.

In [ ]:
rows = []
for rd in ["start", "centre"]:
    B_CO = binarize_sessions(CO, (0, 5), True, 12, rd)
    for s, B in zip(CO, B_CO):
        tr = s["trials"]
        for g in GROUPS:
            m = (s["cells"].projection == g).to_numpy()
            if not m.any():
                continue
            for c in [0.001, 0.01, 0.1]:
                sel = s["keep"] & (tr.conc == c).to_numpy()
                b = B[m][:, sel]
                odors = tr[sel].stim_name.to_numpy()
                r = np.stack([b[:, odors == o].mean(1) for o in np.unique(odors)], 1)
                ls = pcx.vinje_gallant(r, 1)
                ps_ = pcx.vinje_gallant(b.T.astype(float), 1)
                rows.append(dict(reading=rd, subject=s["meta"]["subject"], group=g, conc=c, pct_active=100 * b.mean(),
                               lifetime_sparseness=np.nanmean(ls), n_ls=int(np.isfinite(ls).sum()), n_cells=int(m.sum()),
                               pop_sparseness=np.nanmean(ps_)))
SP = pd.DataFrame(rows)
for rd in ["start", "centre"]:
    summ = SP[SP.reading == rd].groupby(["group", "conc"])[["pct_active", "lifetime_sparseness", "pop_sparseness"]].mean().unstack("conc")
    print(f"reading {rd}")
    print(summ.round(3).to_string())
    fails = []
    for meas in ["pct_active", "lifetime_sparseness", "pop_sparseness"]:
        for g in GROUPS:
            v = summ.loc[g, meas].to_numpy()
            mono = np.all(np.diff(v) > 0) or np.all(np.diff(v) < 0)
            big = abs(v[-1] - v[0]) > 0.5 * abs(v[0])
            print(f"   {meas:20s} {g:9s} values {np.round(v, 3)}, monotonic {mono}, change {100 * (v[-1] - v[0]) / v[0]:+.1f}% of 0.1% value")
            if mono and big:
                fails.append((meas, g))
    print("   populations with a monotonic change of more than half:", fails)

**Result: reproduced.** Under the criterion's reading of "only modestly", no population shows a monotonic change of more than half its 0.1% value, under either window reading. The largest change is the untagged percentage of active cells, 7.3% to 10.8% (+47.9%, start reading) and 7.1% to 10.6% (+48.1%, centre); OB-p active cells rise by 34% and 37%, and lifetime and population sparseness change by 12% or less.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| dataset-neuron-counts | partially reproduced (Go/NoGo OB-p count per session 103 to 108, not 95 +/- 2) |
| fast-dataset-counts | reproduced |
| figs2b-labeling-similar | reproduced |
| mlp-projection-f1, transformer-projection-f1, fig1g-attention-timing, fig1f-latent-separation | blocked: methods underspecified (Tables 5, 6 and 8 missing) |
| fig2b-lifetime-sparseness-similar | not reproduced (mPFC-p and untagged differ) |
| fig2c-mutual-information | not reproduced (OB-p not lower than untagged) |
| fig2d-population-sparseness | not reproduced (all three pairs differ) |
| fig2-proportion-responding | reproduced |
| fig2e-reliability | not reproduced (reliable cells hold, unreliable do not) |
| fig2f-latency | reproduced |
| fig2g-early-peaks | reproduced |
| fig2h-identity-decoding-5s | blocked: methods underspecified (the two windows disagree) |
| fig2i-identity-decoding-early | reproduced |
| fig2j-identity-decoding-peaks-earlier | reproduced (peak times poorly determined) |
| fig2k-mpfc-late-identity | reproduced |
| fig3bc-concentration-tuning | reproduced |
| fig3de-concentration-decoding | partially reproduced (mPFC-p not close to chance) |
| figs5g-pairwise-concentration | not reproduced |
| figs5-sparseness-concentration | reproduced |

How the analysis differs from the paper's. Normalization, cell selection and trial
segmentation follow the paper's code; the figure analyses are not in the code and follow the
Methods and the claims file's readings. The binarization was run under both readings of the
window placement, which give the same outcomes here. The 2.33 SD reliability rule, the
Vinje-Gallant sparseness and the mutual information are the claims file's readings of formulas
lost from the text. Decoding uses 5-fold stratified cross-validation with a new fold split and
cell sample in each repeat. The transformer and MLP claims were not run.

What was found about the dataset. The stated imaging rate is wrong in 21 files (30, 50 or
100 Hz where the timestamps give 4.51 to 4.52 Hz). With the code's rule of skipping planes
without red cells, the counts match the paper's to within 2% for the identity and
concentration experiments, and exactly for the 30 Hz experiment; over all planes the identity
untagged count is 2328 against the paper's 1978. The dandiset has 18 subjects (10 OB-tagged,
8 mPFC-tagged); Figure S2B has 9 OB-p and 10 mPFC-p mice.